# Infomap Brain Network Detection

This notebook extracts time series from preprocessed fMRI data, computes a functional connectivity matrix, and runs Infomap to detect brain networks.

## Download the Glasser atlas (HCP MMP 1.0) in fsLR space

You can download the Glasser atlas (HCP_MMP1.0) in CIFTI `.dlabel.nii` format from the HCP website or via direct link:

- [HCP_MMP1.0 parcellation (Glasser et al., 2016)](https://balsa.wustl.edu/WN56)
- Direct download: [Q1-Q6_RelatedParcellation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors.dlabel.nii](https://balsa.wustl.edu/file/show/9jNk)

After downloading, place the file in a known location and update the `atlas_img_path` in the code below.

In [ ]:
# Step 1: Extract time series from preprocessed fMRI (CIFTI format)
# Ensure nilearn is updated to the latest version
%pip install --upgrade nilearn

from nilearn.maskers import CiftiLabelsMasker
import numpy as np

# Path to preprocessed resting state data (CIFTI dtseries)
fmri_img_path = '/ptmp/hmueller2/Downloads/fmriprep_out/sub-01/ses-15/func/sub-01_ses-15_task-RestingState_dir-ap_space-fsLR_den-91k_bold.dtseries.nii'
# Path to Glasser atlas (fsLR CIFTI dlabel, 32k)
atlas_img_path = '/home/hmueller2/Downloads/atlas_glasser/glasser_hcp/Glasser_et_al_2016_HCP_MMP1.0_v6_RVVG/Q1-Q6_RelatedParcellation210/MNINonLinear/fsaverage_LR32k/Q1-Q6_RelatedParcellation210.CorticalAreas_dil_Colors_210P_Orig.32k_fs_LR.dlabel.nii'  # Update this path

# Extract time series using CiftiLabelsMasker (Nilearn >=0.10 required)
masker = CiftiLabelsMasker(labels_img=atlas_img_path, standardize=True)
time_series = masker.fit_transform(fmri_img_path)
print('Extracted time series shape:', time_series.shape)
# Note: Make sure your atlas matches the space and resolution of your dtseries file.

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


ImportError: cannot import name 'CiftiLabelsMasker' from 'nilearn.maskers' (/home/hmueller2/.local/lib/python3.12/site-packages/nilearn/maskers/__init__.py)

In [ ]:
# Step 2: Compute functional connectivity matrix
%pip install --upgrade nilearn

from nilearn.connectome import ConnectivityMeasure

correlation_measure = ConnectivityMeasure(kind='correlation')
correlation_matrix = correlation_measure.fit_transform([time_series])[0]
print('Correlation matrix shape:', correlation_matrix.shape)

ImportError: cannot import name 'find_stack_level' from 'nilearn._utils.logger' (/home/hmueller2/.local/lib/python3.12/site-packages/nilearn/_utils/logger.py)

In [ ]:
# Step 3: Threshold the matrix to create a network (e.g., keep top 10% edges)
import networkx as nx

threshold = np.percentile(np.abs(correlation_matrix[np.triu_indices_from(correlation_matrix, 1)]), 90)
adjacency = (np.abs(correlation_matrix) >= threshold).astype(int)
np.fill_diagonal(adjacency, 0)

# Create NetworkX graph
G = nx.from_numpy_array(adjacency)
print('Number of nodes:', G.number_of_nodes())
print('Number of edges:', G.number_of_edges())

In [ ]:
# Step 4: Export edge list for Infomap
edge_list_path = 'network_edgelist.txt'
nx.write_edgelist(G, edge_list_path, data=False)
print(f'Edge list saved to {edge_list_path}')

## Step 5: Run Infomap

Run Infomap from the command line (replace with your Infomap binary path):
```
./Infomap network_edgelist.txt . --directed 0 --two-level
```
This will produce a `.tree` file with community assignments.

In [ ]:
# Step 6: (Optional) Load and visualize Infomap results
# ...add code here to parse Infomap output and visualize communities...